In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/2 project ')


In [13]:
df.head()

,Order_ID,Order_Date,Customer,City,Product,Category,Qty,Price,Discount,Payment,Status
0,ORD10002,2026-01-03,Rahul,Jaipur,Shoes,Fashion,8,13441.0,5,UPI,Cancelled
1,ORD10003,2025-11-24,NaN,NaN,Watch,Fashion,1,41478.0,0,NetBanking,Cancelled
2,ORD10004,2025-10-31,Ankit,Mumbai,Chair,Furniture,7,NaN,110,NaN,Pending
3,ORD10005,2025-09-19,Riya,Chandigarh,Table,Furniture,3,622.0,5,Cash,Cancelled
4,ORD10006,2025-05-29,NaN,NaN,Laptop,Electronics,5,11445.0,20,NetBanking,Pending


In [14]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Order_ID    948 non-null    object 
 1   Order_Date  1000 non-null   object 
 2   Customer    823 non-null    object 
 3   City        744 non-null    object 
 4   Product     1000 non-null   object 
 5   Category    1000 non-null   object 
 6   Qty         1000 non-null   int64  
 7   Price       949 non-null    float64
 8   Discount    1000 non-null   int64  
 9   Payment     809 non-null    object 
 10  Status      802 non-null    object 
dtypes: float64(1), int64(2), object(8)
memory usage: 86.1+ KB
None


In [15]:
print(df.describe())

              Qty         Price     Discount
count  1000.00000    949.000000  1000.000000
mean      4.98500  25069.353003    29.970000
std       3.48237  13915.472245    35.870403
min      -5.00000    528.000000     0.000000
25%       3.00000  13245.000000     5.000000
50%       5.00000  24704.000000    15.000000
75%       8.00000  37143.000000    50.000000
max      10.00000  49979.000000   110.000000


In [17]:
print("Missing items check:/n",df.isnull().sum())

Missing items check:/n Order_ID       52
Order_Date      0
Customer      177
City          256
Product         0
Category        0
Qty             0
Price          51
Discount        0
Payment       191
Status        198
dtype: int64


In [25]:
df['Customer']=df['Customer'].fillna(df['Customer'].mode()[0])

Remove extra spaces

In [27]:
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include='object'):
    df[col] = df[col].astype(str).str.strip()

 Remove extra space

In [28]:
df['Customer'] = df['Customer'].fillna('Unknown')

Handle missing values

In [29]:
df['City'] = df['City'].fillna('Unknown')

In [30]:
df['Payment'] = df['Payment'].fillna(df['Payment'].mode()[0])

In [31]:
df['Status'] = df['Status'].fillna('Pending')

In [32]:
df['Price'] = df['Price'].fillna(df['Price'].median())

Handle missing Order ID

In [33]:
df = df.dropna(subset=['Order_ID'])

In [34]:
mask = df['Order_ID'].isna()

df.loc[mask, 'Order_ID'] = [
    f"ORD_NEW_{i}"
    for i in range(mask.sum())
]

Convert date column

In [35]:
df['Order_Date'] = pd.to_datetime(
    df['Order_Date'],
    errors='coerce'
)

In [36]:
df[df['Order_Date'].isna()]

,Order_ID,Order_Date,Customer,City,Product,Category,Qty,Price,Discount,Payment,Status
15,ORD10017,NaT,Karan,Jaipur,Chair,Furniture,3,22160.0,50,Cash,Returned
18,ORD10020,NaT,Amit,nan,Table,Furniture,6,7747.0,0,nan,Pending
43,ORD10045,NaT,Ankit,nan,Shoes,Fashion,8,5434.0,20,UPI,Returned
83,ORD10085,NaT,Amit,nan,Shoes,Fashion,4,6052.0,20,Cash,Delivered
88,nan,NaT,Neha,nan,Chair,Furniture,7,27824.0,50,NetBanking,Returned
105,ORD10107,NaT,Sneha,Jaipur,Shoes,Fashion,9,4045.0,0,NetBanking,Pending
127,ORD10129,NaT,Rahul,nan,Phone,Electronics,1,13079.0,10,NetBanking,Cancelled
129,ORD10131,NaT,Riya,Mumbai,Chair,Furniture,3,16874.0,5,UPI,Pending
145,ORD10147,NaT,Rahul,Delhi,Watch,Fashion,7,11568.0,10,nan,Cancelled
146,ORD10148,NaT,Ankit,Patiala,Chair,Furniture,2,20496.0,20,NetBanking,Returned


Fix quantity values

In [37]:
df = df[df['Qty'] > 0]

In [38]:
df['Qty'] = df['Qty'].abs()

Fix discount values

In [39]:
df['Discount'] = df['Discount'].clip(0, 100)

In [40]:
df = df.drop_duplicates()

Standardize text values

In [41]:
df['Customer'] = df['Customer'].str.title()
df['City'] = df['City'].str.title()
df['Payment'] = df['Payment'].str.title()
df['Status'] = df['Status'].str.title()

Create a sales amount column

In [42]:
df['Sales_Amount'] = (
    df['Qty'] * df['Price']
) * (1 - df['Discount']/100)

Verify cleaned data

In [43]:
print(df.isnull().sum())
print(df.describe())

Order_ID         0
Order_Date      47
Customer         0
City             0
Product          0
Category         0
Qty              0
Price            0
Discount         0
Payment          0
Status           0
Sales_Amount     0
dtype: int64
                       Order_Date         Qty         Price    Discount  \
count                         897  944.000000    944.000000  944.000000   
mean   2025-09-06 08:28:53.779264    5.474576  25145.907839   28.273305   
min           2025-01-01 00:00:00    1.000000    528.000000    0.000000   
25%           2025-05-05 00:00:00    3.000000  13991.250000    5.000000   
50%           2025-09-02 00:00:00    5.000000  24704.000000   15.000000   
75%           2026-01-15 00:00:00    8.000000  36499.000000   50.000000   
max           2026-05-16 00:00:00   10.000000  49979.000000  100.000000   
std                           NaN    2.902697  13570.693240   32.523513   

        Sales_Amount  
count     944.000000  
mean    98489.099470  
min         0.

Save cleaned dataset

In [45]:
df.to_csv(
    "Cleaned_Sales_Data.csv",
    index=False
)